# EfficientNetB0 — Cross-Database Deepfake Detection Experiment

**Research question:** Does a model trained on one deepfake dataset generalize to others?

**Experiment design:**
- Train EfficientNetB0 **from scratch** on each database individually (4 training runs)
- Evaluate on **all 4 databases** per trained model (within-database + cross-database)
- Log all training curves and evaluation metrics to **TensorBoard**
- Display a 4×4 cross-database performance matrix at the end

**Databases:** OpenForensics · CustomWar · CelebDF · CiFake

**Training strategy:** Stage 1 — frozen EfficientNetB0 backbone (ImageNet weights), train head only.  
Stage 2 — unfreeze top 30% of backbone, fine-tune end-to-end at low LR.

In [ ]:
# =================== CELL 1: SETUP ===================
import os
import gc
import glob
import random
import pickle
import numpy as np
import tensorflow as tf
import mimetypes
from datetime import datetime

# Disable XLA/JIT — prevents libdevice errors on this server
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['XLA_FLAGS'] = '--xla_gpu_cuda_data_dir=/usr/lib/nvidia-cuda-toolkit'
tf.config.optimizer.set_jit(False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print('✅ GPU memory growth enabled')
    except RuntimeError as e:
        print(e)

tf.keras.backend.clear_session()
print(f'✅ Setup complete — TensorFlow {tf.__version__}')

In [ ]:
# =================== CELL 2: PATHS & TENSORBOARD CONFIG ===================

GDRIVE_PATH   = os.path.expanduser('~/RealEyes/gdrive')
DATASET_ROOT  = os.path.join(GDRIVE_PATH, 'data_set_split')       # CustomWar split
DATASETS_DIR  = os.path.expanduser('~/RealEyes/RealEyes/datasets')
CELEBDF_DIR   = os.path.join(DATASETS_DIR, 'celebdf_v2')

MODEL_NAME            = 'efficientnet'
EXPERIMENT_MODELS_DIR = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/no_lora_experiment'  # no-LoRA run
EXPERIMENT_MODELS_DIR = '/home/sceuser/RealEyes/gdrive/deepfake_image_project/models/no_lora_experiment'  # no-LoRA run
os.makedirs(MODEL_DIR, exist_ok=True)

TB_LOG_ROOT = os.path.expanduser('~/RealEyes/tensorboard_logs')
os.makedirs(TB_LOG_ROOT, exist_ok=True)


def get_tb_log_dir(train_db_name, suffix='train'):
    ts = datetime.now().strftime('%Y%m%d_%H%M')
    return os.path.join(TB_LOG_ROOT, MODEL_NAME, f'{suffix}_{train_db_name}', ts)


def log_eval_to_tb(train_db_name, test_db_name, metrics: dict, step=0):
    """Write one evaluation result (cross-db or within-db) to TensorBoard."""
    log_dir = os.path.join(
        TB_LOG_ROOT, MODEL_NAME, 'cross_db_eval',
        f'train_{train_db_name}__test_{test_db_name}'
    )
    writer = tf.summary.create_file_writer(log_dir)
    with writer.as_default():
        for name, value in metrics.items():
            tf.summary.scalar(name, float(value), step=step)
    writer.flush()


print('✅ Paths ready')
print(f'  MODEL_DIR  : {MODEL_DIR}')
print(f'  TB_LOG_ROOT: {TB_LOG_ROOT}')
print()
print('▶  View TensorBoard:')
print(f'  [server ] tensorboard --logdir {TB_LOG_ROOT} --port 6006 --bind_all')
print( '  [local  ] ssh -L 6006:localhost:6006 <user>@<server_ip>')
print( '  [browser] http://localhost:6006')

In [ ]:
# =================== CELL 3: DATABASE DEFINITIONS ===================
# Each database entry: {'train': path, 'val': path, 'test': path}
# CiFake has no independent test split, so test == val.

DATABASES = {}


def _try_add(name, paths):
    missing = [k for k, v in paths.items() if not os.path.isdir(v)]
    if missing:
        print(f'  ⚠️  {name}: missing splits {missing} — skipped')
        return
    DATABASES[name] = paths
    print(f'  ✅ {name}')


print('🗂  Scanning databases...')

_try_add('OpenForensics', {
    'train': os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Train'),
    'val':   os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Validation'),
    'test':  os.path.join(DATASETS_DIR, 'OpenForensicsV1/Dataset/Test'),
})
_try_add('CustomWar', {
    'train': os.path.join(DATASET_ROOT, 'train'),
    'val':   os.path.join(DATASET_ROOT, 'val'),
    'test':  os.path.join(DATASET_ROOT, 'test'),
})
_try_add('CelebDF', {
    'train': os.path.join(CELEBDF_DIR, 'train'),
    'val':   os.path.join(CELEBDF_DIR, 'val'),
    'test':  os.path.join(CELEBDF_DIR, 'test'),
})
_try_add('CiFake', {
    'train': os.path.join(DATASETS_DIR, 'cifake/train'),
    'val':   os.path.join(DATASETS_DIR, 'cifake/test'),
    'test':  os.path.join(DATASETS_DIR, 'cifake/test'),
})

print(f'\n📋 Active databases: {list(DATABASES.keys())}')

In [ ]:
# =================== CELL 4: DATA LOADING HELPERS ===================

def load_dataset_images(dataset_path, max_images=None):
    """Load image paths and binary labels (REAL=0, FAKE=1) from a REAL/FAKE directory."""
    valid_ext = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}
    image_paths, labels, skipped = [], [], 0

    for folder in sorted(os.listdir(dataset_path)):
        fpath = os.path.join(dataset_path, folder)
        if not os.path.isdir(fpath):
            continue
        fu = folder.upper()
        if fu == 'FAKE':
            label = 1
        elif fu == 'REAL':
            label = 0
        else:
            print(f'  ⚠️  Unknown folder "{folder}" in {dataset_path} — skipped')
            continue

        collected = []
        for root, _, files in os.walk(fpath):
            for fname in files:
                if os.path.splitext(fname)[1].lower() not in valid_ext:
                    skipped += 1
                    continue
                collected.append(os.path.join(root, fname))
        if max_images:
            collected = collected[:max_images]
        image_paths.extend(collected)
        labels.extend([label] * len(collected))

    if skipped:
        print(f'  ⚠️  {skipped} non-image files skipped')
    return np.array(image_paths), np.array(labels)


def load_db_split(db_name, split='train'):
    paths, labels = load_dataset_images(DATABASES[db_name][split])
    n_real = int(np.sum(labels == 0))
    n_fake = int(np.sum(labels == 1))
    print(f'    {db_name}/{split}: {len(paths):,} images  (REAL={n_real:,}, FAKE={n_fake:,})')
    return paths, labels


def compute_class_weights(labels):
    from sklearn.utils.class_weight import compute_class_weight
    classes = np.unique(labels)
    weights = compute_class_weight('balanced', classes=classes, y=labels)
    return {int(c): float(w) for c, w in zip(classes, weights)}


print('✅ Data loading helpers ready')

In [ ]:
# =================== CELL 5: EFFICIENTNET DATASET PIPELINE ===================

from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

IMG_SIZE = (224, 224)
AUTOTUNE = tf.data.AUTOTUNE


def _decode(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, tf.cast(label, tf.float32)


def _augment(img, label):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.15)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
    img = tf.clip_by_value(img, 0.0, 255.0)
    return img, label


def build_eff_dataset(image_paths, labels, batch_size=32, training=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if training:
        ds = ds.shuffle(min(len(image_paths), 10000), reshuffle_each_iteration=True)
    ds = ds.map(_decode, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
    ds = ds.map(lambda x, y: (eff_preprocess(x), y), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(4)
    options = tf.data.Options()
    options.autotune.ram_budget = 3 * 1024 * 1024 * 1024
    ds = ds.with_options(options)
    return ds


print('✅ EfficientNet dataset pipeline ready')

In [ ]:
# =================== CELL 6: EFFICIENTNET MODEL BUILDER ===================

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model
from tensorflow.keras.metrics import AUC, Precision, Recall


def build_efficientnet_b0(train_base=False):
    """EfficientNetB0 with ImageNet weights + custom binary classification head."""
    base = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3)
    )
    base.trainable = train_base

    inputs  = layers.Input(shape=(224, 224, 3), name='rgb_input')
    x       = base(inputs, training=False)
    x       = layers.GlobalAveragePooling2D(name='gap')(x)
    x       = layers.Dropout(0.30, name='drop1')(x)
    x       = layers.Dense(256, activation='relu', name='dense1')(x)
    x       = layers.Dropout(0.25, name='drop2')(x)
    x       = layers.Dense(64,  activation='relu', name='dense2')(x)
    x       = layers.Dropout(0.15, name='drop3')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='prob_fake')(x)

    model = Model(inputs, outputs, name='EfficientNetB0_Deepfake')
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.BinaryCrossentropy(),
        metrics=['accuracy', AUC(name='auc'), Precision(name='precision'), Recall(name='recall')]
    )
    return model


print('✅ EfficientNetB0 builder ready')
# Quick parameter count
_tmp = build_efficientnet_b0()
print(f'  Total parameters: {_tmp.count_params():,}')
del _tmp
tf.keras.backend.clear_session()

In [ ]:
# =================== CELL 7: EVALUATION HELPERS ===================

from sklearn.metrics import classification_report, roc_auc_score, accuracy_score


def evaluate_model(model, test_ds):
    """Run model on test_ds and return (metrics_dict, classification_report_dict)."""
    y_true, y_prob = [], []
    for x_batch, y_batch in test_ds:
        preds = model.predict(x_batch, verbose=0).flatten()
        y_true.extend(y_batch.numpy())
        y_prob.extend(preds)

    y_true = np.array(y_true).astype(int)
    y_prob = np.array(y_prob)
    y_pred = (y_prob > 0.5).astype(int)

    report = classification_report(
        y_true, y_pred, target_names=['REAL', 'FAKE'], output_dict=True, digits=4
    )
    metrics = {
        'accuracy':       float(accuracy_score(y_true, y_pred)),
        'roc_auc':        float(roc_auc_score(y_true, y_prob)),
        'f1_fake':        float(report['FAKE']['f1-score']),
        'f1_real':        float(report['REAL']['f1-score']),
        'precision_fake': float(report['FAKE']['precision']),
        'recall_fake':    float(report['FAKE']['recall']),
    }
    return metrics, report


def print_eval_report(train_db, test_db, metrics, report):
    tag = '  ✅ [WITHIN-DB]' if train_db == test_db else '  🔀 [CROSS-DB ]'
    print(f'\n{tag}  Trained on {train_db:<15} → Tested on {test_db}')
    sep = '  ' + '─' * 58
    print(sep)
    for cls in ['REAL', 'FAKE']:
        r = report[cls]
        print(f'  {cls:<6}  P={r["precision"]:.4f}  R={r["recall"]:.4f}  F1={r["f1-score"]:.4f}  n={int(r["support"]):,}')
    print(sep)
    print(f'  Accuracy={metrics["accuracy"]:.4f}   ROC-AUC={metrics["roc_auc"]:.4f}')


print('✅ Evaluation helpers ready')

In [ ]:
import ctypes
import traceback
# =================== CELL 8: MAIN EXPERIMENT LOOP ===================
#
# For each training database:
#   Stage 1 — train classification head, backbone frozen (ImageNet weights)
#   Stage 2 — unfreeze top 30% of backbone, fine-tune at low LR
#   Evaluate — test on ALL 4 databases (within + cross), log to TensorBoard
# ─────────────────────────────────────────────────────────────────────────

BATCH_SIZE  = 32
all_results = {}   # all_results[train_db][test_db] = metrics_dict

for train_db_name in DATABASES:
    print(f'\n{"="*70}')
    print(f'  🎯  TRAINING ON: {train_db_name}')
    print(f'{"="*70}')

    # ── 1. Load training data ─────────────────────────────────────────────
    print('\n📦 Loading training data...')
    train_paths, train_lbls = load_db_split(train_db_name, 'train')
    val_paths,   val_lbls   = load_db_split(train_db_name, 'val')
    class_weights           = compute_class_weights(train_lbls)
    print(f'  Class weights: {class_weights}')

    train_ds = build_eff_dataset(train_paths, train_lbls, batch_size=BATCH_SIZE, training=True)
    val_ds   = build_eff_dataset(val_paths,   val_lbls,   batch_size=BATCH_SIZE, training=False)

    # ── 2. Build fresh model ──────────────────────────────────────────────
    gc.collect()
    tf.keras.backend.clear_session()
    model   = build_efficientnet_b0(train_base=False)
    tb_log  = get_tb_log_dir(train_db_name, suffix='train')
    s1_path = os.path.join(MODEL_DIR, f'eff_stage1_{train_db_name}.keras')

    # ── 3. Stage 1: frozen backbone ───────────────────────────────────────
    print(f'\n🚀 Stage 1 — frozen backbone ({train_db_name})...')
    callbacks_s1 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=5,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5,
            patience=2, min_lr=1e-6, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            s1_path, monitor='val_auc', mode='max',
            save_best_only=True, verbose=1),
        tf.keras.callbacks.TensorBoard(
            log_dir=tb_log, histogram_freq=0, update_freq='epoch'),
    ]
    model.fit(
        train_ds, validation_data=val_ds,
        epochs=15, class_weight=class_weights,
        callbacks=callbacks_s1, verbose=1
    )

    # ── 4. Stage 2: partial unfreeze ──────────────────────────────────────
    print(f'\n🚀 Stage 2 — unfreeze top 30% of backbone ({train_db_name})...')
    model    = tf.keras.models.load_model(s1_path, compile=False)
    eff_base = model.get_layer('efficientnetb0')
    eff_base.trainable = True
    fine_tune_at = int(len(eff_base.layers) * 0.70)
    for lyr in eff_base.layers[:fine_tune_at]:
        lyr.trainable = False
    print(f'  Backbone layers: {len(eff_base.layers)} | Frozen: {fine_tune_at} | Trainable: {len(eff_base.layers)-fine_tune_at}')

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5, clipnorm=1.0),
        loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=0.05),
        metrics=['accuracy', AUC(name='auc'), Precision(name='precision'), Recall(name='recall')]
    )

    best_path = os.path.join(MODEL_DIR, f'trained_on_{train_db_name}.keras')
    callbacks_s2 = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_auc', mode='max', patience=6,
            restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_auc', mode='max', factor=0.5,
            patience=3, min_lr=1e-7, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(
            best_path, monitor='val_auc', mode='max',
            save_best_only=True, verbose=1),
        tf.keras.callbacks.TensorBoard(
            log_dir=tb_log, histogram_freq=0, update_freq='epoch'),
    ]
    model.fit(
        train_ds, validation_data=val_ds,
        epochs=15, class_weight=class_weights,
        callbacks=callbacks_s2, verbose=1
    )

    # ── 5. Evaluate on ALL databases ──────────────────────────────────────
    print(f'\n\n📊 EVALUATION — model trained on {train_db_name}')
    best_model = tf.keras.models.load_model(best_path, compile=False)
    all_results[train_db_name] = {}

    for test_db_name in DATABASES:
        print(f'\n  🔍 Test database: {test_db_name}...')
        test_paths, test_lbls = load_db_split(test_db_name, 'test')
        test_ds               = build_eff_dataset(test_paths, test_lbls, batch_size=BATCH_SIZE, training=False)
        metrics, report       = evaluate_model(best_model, test_ds)
        all_results[train_db_name][test_db_name] = metrics
        log_eval_to_tb(train_db_name, test_db_name, metrics)
        print_eval_report(train_db_name, test_db_name, metrics, report)

    # ── 6. Memory cleanup ─────────────────────────────────────────────────
    del model, best_model, train_ds, val_ds
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass
    tf.keras.backend.clear_session()
    print(f'\n✅ {train_db_name} experiment complete — GPU memory cleared.')

# Persist results for cross-model comparison later
results_path = os.path.join(MODEL_DIR, 'all_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f'\n\n{"="*70}')
print('  🏁  ALL EXPERIMENTS COMPLETE')
print(f'{"="*70}')
print(f'\nResults saved → {results_path}')

In [ ]:
# =================== CELL 9: CROSS-DATABASE RESULTS MATRIX ===================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

db_names = list(DATABASES.keys())

for metric_key, metric_label, cmap in [
    ('roc_auc',  'ROC-AUC',  'YlOrRd'),
    ('accuracy', 'Accuracy', 'YlGn'),
    ('f1_fake',  'F1-FAKE',  'Blues'),
]:
    available = [d for d in db_names if d in all_results]
    matrix = [
        [all_results[tr].get(te, {}).get(metric_key, float('nan')) for te in db_names]
        for tr in available
    ]
    df = pd.DataFrame(
        matrix,
        index=[f'Train: {d}' for d in available],
        columns=[f'Test: {d}' for d in db_names]
    )

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.heatmap(
        df.astype(float), annot=True, fmt='.3f',
        cmap=cmap, vmin=0.40, vmax=1.00, ax=ax,
        linewidths=0.5, linecolor='gray',
        cbar_kws={'label': metric_label}
    )
    ax.set_title(
        f'EfficientNetB0 — Cross-Database {metric_label}\n'
        '(diagonal = within-database · off-diagonal = cross-database)',
        fontsize=13, fontweight='bold'
    )
    ax.set_ylabel('Training Database', fontsize=11)
    ax.set_xlabel('Test Database', fontsize=11)
    plt.xticks(rotation=30, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    fig_path = os.path.join(MODEL_DIR, f'cross_db_{metric_key}.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {fig_path}')
    print(f'\n{metric_label} Matrix:')
    print(df.to_string())
    print()

In [ ]:
# =================== CELL 10: TENSORBOARD LAUNCH INSTRUCTIONS ===================

print('━' * 60)
print('  📊  TENSORBOARD DASHBOARD')
print('━' * 60)
print()
print('1️⃣  On the SERVER terminal, run:')
print(f'   tensorboard --logdir {TB_LOG_ROOT} --port 6006 --bind_all')
print()
print('2️⃣  On your LOCAL machine, open an SSH tunnel:')
print('   ssh -L 6006:localhost:6006 <your_user>@<server_ip>')
print()
print('3️⃣  Open in browser:')
print('   http://localhost:6006')
print()
print('What you will see:')
print('  SCALARS tab  → training loss/accuracy/auc per epoch, per database')
print('  cross_db_eval → roc_auc/accuracy/f1 for every (train→test) pair')
print()
print('TensorBoard log structure:')
print(f'  {TB_LOG_ROOT}/')
print(f'  └── {MODEL_NAME}/')
print(f'      ├── train_OpenForensics/   ← training curves')
print(f'      ├── train_CustomWar/')
print(f'      ├── train_CelebDF/')
print(f'      ├── train_CiFake/')
print(f'      └── cross_db_eval/')
print(f'          ├── train_OpenForensics__test_CustomWar/')
print(f'          ├── train_OpenForensics__test_CelebDF/')
print(f'          └── ...')